وظيفة لغز عبور الجسر

تعريف المكتبة والحقائق المستخدمة

In [ ]:
from experta import*

class State(Fact):
    pass

تعريف الحالة الابتدائية و DefFacts()

In [ ]:
class BridgeExpertSystem(KnowledgeEngine):
    
    @DefFacts()
    def initial_state(self):
        yield State(
            left=('me', 'lab', 'worker', 'scientist'),
            right=(),
            light='left',
            time=0,
            path=[]
        )

قواعد توليد ابناء الحالة

In [ ]:
#قاعدة عبور شخصين من اليسار لليمين
@Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path))
def move_left_to_right(self, left, right, time, path):
    from itertools import combinations

    persons_times = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}

    for p1, p2 in combinations(left, 2):
        new_left = list(left)
        new_left.remove(p1)
        new_left.remove(p2)
        new_right = list(right) + [p1, p2]

        t = max(persons_times[p1], persons_times[p2])
        total_time = time + t

        if total_time <= 17:
            self.declare(State(
                left=tuple(new_left),
                right=tuple(new_right),
                light='right',
                time=total_time,
                path=path + [f"{p1} and {p2} crossed to right in {t} min"]
            ))


قواعد التحقق من الشروط

In [ ]:
#قاعدة العودة من اليمين الى اليسار لشخص واحد, عودة المصباح من اليمين الى اليسار
@Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path))
def move_right_to_left(self, left, right, time, path):
    persons_times = {'you': 1, 'lab': 2, 'worker': 5, 'scientist': 10}

    for p in right:
        new_right = list(right)
        new_right.remove(p)
        new_left = list(left) + [p]

        t = persons_times[p]
        total_time = time + t

        if total_time <= 17:
            self.declare(State(
                left=tuple(new_left),
                right=tuple(new_right),
                light='left',
                time=total_time,
                path=path + [f"{p} returned to left in {t} min"]
            ))


قاعدة التحقق من الوصول الى الحالة الهدف

قواعد طباعة الحل

تشغيل الخبير

In [ ]:
engine = BridgeExpertSystem()
engine.reset()
engine.run()


In [29]:
from experta import *

class State(Fact):
    salience = Field(int, default=900)

class BaseBridgeExpertSystem(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.persons_time = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        self.max_time = 17
        self.visited = set()
        self.tree = {}
        self.node_counter = 0

    @DefFacts()
    def _initial_state(self):
        yield State(left=('me', 'lab', 'worker', 'scientist'),
                    right=(),
                    light='left',
                    time=0,
                    path=(),
                    node=0,
                    parent=None,
                    depth=0)

    def generate_state(self, left_list, right_list, light, time, path, step, parent, depth):
        signature = (tuple(sorted(left_list)), tuple(sorted(right_list)), light, time)
        if signature in self.visited:
            return
        self.visited.add(signature)

        self.node_counter += 1
        node = self.node_counter
        self.tree[node] = {'parent': parent, 'left': tuple(left_list), 'right': tuple(right_list),
                           'light': light, 'time': time, 'depth': depth}
        new_path = list(path) + [step]
        self.declare(State(left=tuple(left_list), right=tuple(right_list), light=light, time=time,
                           path=tuple(new_path), node=node, parent=parent, depth=depth))

    def move_pair(self, left, right, time, path, node, depth, p1, p2):
        if p1 in left and p2 in left:
            new_left = list(left)
            new_right = list(right)
            new_left.remove(p1)
            new_left.remove(p2)
            new_right += [p1, p2]
            duration = max(self.persons_time[p1], self.persons_time[p2])
            new_time = time + duration
            if new_time <= self.max_time:
                step = f"{p1} and {p2} crossed to right in {duration} min"
                self.generate_state(new_left, new_right, 'right', new_time, path, step, node, depth + 1)

    def return_one(self, left, right, time, path, node, depth, p):
        if p in right:
            new_left = list(left)
            new_right = list(right)
            new_right.remove(p)
            new_left.append(p)
            duration = self.persons_time[p]
            new_time = time + duration
            if new_time <= self.max_time:
                step = f"{p} returned to left in {duration} min"
                self.generate_state(new_left, new_right, 'left', new_time, path, step, node, depth + 1)

    @Rule(State(left=MATCH.left, right=MATCH.right, time=MATCH.time, path=MATCH.path),
          TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
          salience=1000)
    def goal_reached(self, left, right, time, path):
        print("\n✅ تم الوصول إلى الهدف في:", time, "دقيقة")
        print("\n📜 خطوات الحل:")
        for i, step in enumerate(path, 1):
            print(f"{i}. {step}")
        print("\n🌳 شجرة البحث:")
        for node_id, data in self.tree.items():
            print(f"🔸 Node {node_id} (Parent: {data['parent']}, Depth: {data['depth']}) | Left: {data['left']} | Right: {data['right']} | Light: {data['light']} | Time: {data['time']}")
        self.halt()


class DFSBridgeExpertSystem(BaseBridgeExpertSystem):
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                    time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                  )
    def move_me_and_lab(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'me', 'lab')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                )
    def move_me_and_worker(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'me', 'worker')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                )
    def move_me_and_scientist(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'me', 'scientist')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
               )
    def move_lab_and_worker(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'lab', 'worker')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                )
    def move_lab_and_scientist(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'lab', 'scientist')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                )
    def move_worker_and_scientist(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'worker', 'scientist')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                    time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                    )
    def return_me(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'me')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                )
    def return_lab(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'lab')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
    )
    def return_worker(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'worker')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                )
    def return_scientist(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'scientist')

class BFSBridgeExpertSystem(BaseBridgeExpertSystem):
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                    time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                     salience=900)
    def move_me_and_lab(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'me', 'lab')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                 salience=800)
    def move_me_and_worker(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'me', 'worker')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                 salience=600)
    def move_me_and_scientist(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'me', 'scientist')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                 salience=700)
    def move_lab_and_worker(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'lab', 'worker')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                salience=500)
    def move_lab_and_scientist(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'lab', 'scientist')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                 salience=400)
    def move_worker_and_scientist(self, left, right, time, path, node, depth):
        self.move_pair(left, right, time, path, node, depth, 'worker', 'scientist')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
            time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
             salience=900)
    def return_me(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'me')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                 salience=800)
    def return_lab(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'lab')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
             salience=700)
    def return_worker(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'worker')

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right',
                time=MATCH.time, path=MATCH.path, node=MATCH.node, depth=MATCH.depth),
                 salience=600)
    def return_scientist(self, left, right, time, path, node, depth):
        self.return_one(left, right, time, path, node, depth, 'scientist')
# --- تشغيل النظام ---
print("\n==== البحث باستخدام DFS ====")
engine_dfs = DFSBridgeExpertSystem()
engine_dfs.reset()
engine_dfs.run()

print("\n==== البحث باستخدام BFS ====")
engine_bfs = BFSBridgeExpertSystem()
engine_bfs.reset()
engine_bfs.run()



==== البحث باستخدام DFS ====

✅ تم الوصول إلى الهدف في: 17 دقيقة

📜 خطوات الحل:
1. me and lab crossed to right in 2 min
2. me returned to left in 1 min
3. worker and scientist crossed to right in 10 min
4. lab returned to left in 2 min
5. me and lab crossed to right in 2 min

🌳 شجرة البحث:
🔸 Node 1 (Parent: 0, Depth: 1) | Left: ('lab', 'scientist') | Right: ('me', 'worker') | Light: right | Time: 5
🔸 Node 2 (Parent: 1, Depth: 2) | Left: ('lab', 'scientist', 'me') | Right: ('worker',) | Light: left | Time: 6
🔸 Node 3 (Parent: 2, Depth: 3) | Left: ('lab',) | Right: ('worker', 'me', 'scientist') | Light: right | Time: 16
🔸 Node 4 (Parent: 3, Depth: 4) | Left: ('lab', 'me') | Right: ('worker', 'scientist') | Light: left | Time: 17
🔸 Node 5 (Parent: 2, Depth: 3) | Left: ('scientist',) | Right: ('worker', 'me', 'lab') | Light: right | Time: 8
🔸 Node 6 (Parent: 5, Depth: 4) | Left: ('scientist', 'me') | Right: ('worker', 'lab') | Light: left | Time: 9
🔸 Node 7 (Parent: 5, Depth: 4) | Left: (